# Quantum Neural Network (QNN)

Notebook này xây dựng một mô hình **Quantum Neural Network** dạng **Variational Quantum Classifier** để phân loại dữ liệu nhị phân 2 chiều.

Mục tiêu của notebook:

1. Giữ code QNN đủ nhỏ để đọc được từ đầu đến cuối.
2. Tách rõ từng khái niệm: dữ liệu, state vector, cổng lượng tử, feature map, ansatz, measurement, loss và gradient.
3. Dùng tên biến/hàm tường minh để notebook có thể tái sử dụng như tài liệu học QNN cơ bản.

Ý tưởng chính:

1. **Mã hóa dữ liệu cổ điển vào trạng thái lượng tử** bằng các cổng quay phụ thuộc vào input.
2. **Dùng mạch lượng tử có tham số** giống như lớp học được của neural network.
3. **Đo observable** để lấy một giá trị dự đoán cổ điển.
4. **Tối ưu tham số** bằng gradient parameter-shift.

Ví dụ dưới đây không cần Qiskit. Ta tự mô phỏng state vector bằng NumPy để nhìn rõ cơ chế bên trong.


## 1. Chuẩn bị thư viện

Ta dùng `numpy` để tính toán vector trạng thái lượng tử, `matplotlib` để trực quan hóa dữ liệu, loss, accuracy, và `scikit-learn` để tải bộ dữ liệu Iris.

Nếu môi trường chưa có thư viện, cài bằng:

```bash
pip install numpy matplotlib scikit-learn
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

RANDOM_SEED = 7
rng = np.random.default_rng(RANDOM_SEED)

plt.rcParams['figure.dpi'] = 110


## 2. Tải tập dữ liệu Iris

Ta dùng bộ dữ liệu **Iris**, gồm các phép đo hình thái của hoa diên vĩ.

Bộ Iris gốc có 3 lớp và 4 đặc trưng:

- `sepal length`
- `sepal width`
- `petal length`
- `petal width`

Mô hình QNN hiện tại là mô hình **phân loại nhị phân** với **2 qubit**, nên ta chọn:

- 2 lớp: `setosa` và `versicolor`.
- 2 đặc trưng: `petal length` và `petal width`.

Hai đặc trưng này được scale về khoảng `[-1, 1]` để phù hợp với feature map lượng tử `RY(pi * x)` và `RZ(pi * x^2)`.


In [ ]:
def load_binary_iris_dataset():
    """Return a 2-feature, 2-class Iris dataset scaled to [-1, 1]."""
    iris = load_iris()

    class_indices = [0, 1]  # 0 = setosa, 1 = versicolor
    feature_indices = [2, 3]  # petal length, petal width

    selected_rows = np.isin(iris.target, class_indices)
    raw_features = iris.data[selected_rows][:, feature_indices]
    labels = iris.target[selected_rows].astype(float)

    feature_min = raw_features.min(axis=0)
    feature_max = raw_features.max(axis=0)
    scaled_features = 2 * (raw_features - feature_min) / (feature_max - feature_min) - 1

    feature_names = [iris.feature_names[index] for index in feature_indices]
    class_names = [iris.target_names[index] for index in class_indices]

    return scaled_features, labels, feature_names, class_names


def plot_dataset(features, labels, feature_names):
    """Plot the two selected Iris features after scaling."""
    plt.figure(figsize=(5, 4))
    plt.scatter(features[:, 0], features[:, 1], c=labels, cmap='coolwarm', edgecolor='k')
    plt.title('Iris dataset: setosa vs versicolor')
    plt.xlabel(f'{feature_names[0]} scaled')
    plt.ylabel(f'{feature_names[1]} scaled')
    plt.grid(alpha=0.25)
    plt.show()


features, labels, feature_names, class_names = load_binary_iris_dataset()

plot_dataset(features, labels, feature_names)

print('Classes:', class_names)
print('Features:', feature_names)
print('features shape:', features.shape)
print('labels shape:', labels.shape)


## 3. Các khối lượng tử cơ bản

Một hệ `n` qubit được biểu diễn bằng vector phức có kích thước `2^n`.

Với 2 qubit, trạng thái tổng quát là:

$$\lvert\psi\rangle = a_{00}\lvert00\rangle + a_{01}\lvert01\rangle + a_{10}\lvert10\rangle + a_{11}\lvert11\rangle$$

Các hệ số `a` là biên độ xác suất và phải thỏa:

$$\sum_i |a_i|^2 = 1$$

Ta sẽ cài đặt các cổng `RY(theta)`, `RZ(theta)`, `CNOT`, và phép đo kỳ vọng `Z` trên qubit đầu tiên để lấy output.

Quy ước index trong notebook:

- `qubit = 0` là qubit bên trái trong trạng thái `|q0 q1>`.
- `qubit = 1` là qubit bên phải.
- Với 2 qubit, thứ tự vector trạng thái là `|00>`, `|01>`, `|10>`, `|11>`.


In [ ]:
def create_zero_state(n_qubits):
    """Create |00...0> as a complex state vector."""
    state = np.zeros(2 ** n_qubits, dtype=complex)
    state[0] = 1.0
    return state


def rotation_y(angle):
    """Return the single-qubit RY rotation matrix."""
    cos_half_angle = np.cos(angle / 2)
    sin_half_angle = np.sin(angle / 2)

    return np.array(
        [[cos_half_angle, -sin_half_angle], [sin_half_angle, cos_half_angle]],
        dtype=complex,
    )


def rotation_z(angle):
    """Return the single-qubit RZ rotation matrix."""
    return np.array(
        [[np.exp(-0.5j * angle), 0], [0, np.exp(0.5j * angle)]],
        dtype=complex,
    )


def qubit_bit_mask(qubit_index, n_qubits):
    """Map a qubit index to the matching bit mask in the state-vector index."""
    return 1 << (n_qubits - 1 - qubit_index)


def apply_single_qubit_gate(state, gate, qubit_index, n_qubits):
    """Apply a 2x2 gate to one qubit of a state vector."""
    next_state = state.copy()
    mask = qubit_bit_mask(qubit_index, n_qubits)

    for basis_index in range(len(state)):
        if basis_index & mask:
            continue

        paired_basis_index = basis_index | mask
        amplitude_zero = state[basis_index]
        amplitude_one = state[paired_basis_index]

        next_state[basis_index] = gate[0, 0] * amplitude_zero + gate[0, 1] * amplitude_one
        next_state[paired_basis_index] = gate[1, 0] * amplitude_zero + gate[1, 1] * amplitude_one

    return next_state


def apply_cnot(state, control_qubit, target_qubit, n_qubits):
    """Apply a CNOT gate: flip target when control is |1>."""
    next_state = state.copy()
    control_mask = qubit_bit_mask(control_qubit, n_qubits)
    target_mask = qubit_bit_mask(target_qubit, n_qubits)

    for basis_index in range(len(state)):
        control_is_one = basis_index & control_mask
        target_is_zero = not (basis_index & target_mask)

        if control_is_one and target_is_zero:
            flipped_basis_index = basis_index | target_mask
            next_state[basis_index] = state[flipped_basis_index]
            next_state[flipped_basis_index] = state[basis_index]

    return next_state


def measure_z_expectation(state, qubit_index, n_qubits):
    """Return <Z> for one qubit; result is in [-1, 1]."""
    mask = qubit_bit_mask(qubit_index, n_qubits)
    expectation = 0.0

    for basis_index, amplitude in enumerate(state):
        probability = np.abs(amplitude) ** 2
        z_eigenvalue = -1 if (basis_index & mask) else 1
        expectation += z_eigenvalue * probability

    return float(np.real(expectation))


## 4. Kiến trúc QNN

Mô hình gồm hai phần.

### 4.1. Feature map

Feature map biến input cổ điển `x = [x1, x2]` thành trạng thái lượng tử:

- `RY(pi * x1)` và `RY(pi * x2)` đưa giá trị input vào từng qubit.
- `RZ(pi * x_i^2)` thêm thông tin phi tuyến cục bộ.
- `CNOT` và `RZ(pi * x1 * x2)` đưa tương tác giữa hai đặc trưng vào mạch.

### 4.2. Variational ansatz

Ansatz là phần có tham số học được. Mỗi layer gồm `RY`, `RZ`, và các cổng `CNOT` để entangle các qubit. Các tham số `theta` đóng vai trò tương tự trọng số trong neural network cổ điển.

Với `N_QUBITS = 2` và `N_LAYERS = 3`, mỗi layer có:

- 2 tham số cho các cổng `RY`.
- 2 tham số cho các cổng `RZ`.
- Tổng cộng `2 * 2 * 3 = 12` tham số học được.


In [ ]:
N_QUBITS = 2
N_LAYERS = 3
PARAMS_PER_QUBIT = 2  # one RY angle and one RZ angle
PARAMS_PER_LAYER = N_QUBITS * PARAMS_PER_QUBIT
N_TRAINABLE_PARAMS = N_LAYERS * PARAMS_PER_LAYER


def build_feature_map(sample):
    """Encode one classical sample into a quantum state."""
    state = create_zero_state(N_QUBITS)

    for qubit_index, feature_value in enumerate(sample):
        state = apply_single_qubit_gate(
            state,
            rotation_y(np.pi * feature_value),
            qubit_index,
            N_QUBITS,
        )
        state = apply_single_qubit_gate(
            state,
            rotation_z(np.pi * feature_value ** 2),
            qubit_index,
            N_QUBITS,
        )

    interaction_angle = np.pi * sample[0] * sample[1]
    state = apply_cnot(state, control_qubit=0, target_qubit=1, n_qubits=N_QUBITS)
    state = apply_single_qubit_gate(state, rotation_z(interaction_angle), 1, N_QUBITS)
    state = apply_cnot(state, control_qubit=0, target_qubit=1, n_qubits=N_QUBITS)

    return state


def apply_variational_ansatz(encoded_state, trainable_params):
    """Apply trainable RY/RZ layers and entangling CNOT gates."""
    state = encoded_state
    param_index = 0

    for _ in range(N_LAYERS):
        for qubit_index in range(N_QUBITS):
            angle = trainable_params[param_index]
            state = apply_single_qubit_gate(state, rotation_y(angle), qubit_index, N_QUBITS)
            param_index += 1

        for qubit_index in range(N_QUBITS):
            angle = trainable_params[param_index]
            state = apply_single_qubit_gate(state, rotation_z(angle), qubit_index, N_QUBITS)
            param_index += 1

        state = apply_cnot(state, control_qubit=0, target_qubit=1, n_qubits=N_QUBITS)
        state = apply_cnot(state, control_qubit=1, target_qubit=0, n_qubits=N_QUBITS)

    return state


def run_qnn(sample, trainable_params):
    """Return <Z> after feature map + ansatz for one sample."""
    encoded_state = build_feature_map(sample)
    final_state = apply_variational_ansatz(encoded_state, trainable_params)
    return measure_z_expectation(final_state, qubit_index=0, n_qubits=N_QUBITS)


def class_one_probability(sample, trainable_params):
    """Convert <Z> from [-1, 1] to P(y = 1) in [0, 1]."""
    z_expectation = run_qnn(sample, trainable_params)
    return (1 - z_expectation) / 2


def predict_labels_and_probabilities(feature_matrix, trainable_params, threshold=0.5):
    """Predict class labels and P(y = 1) for many samples."""
    probabilities = np.array([
        class_one_probability(sample, trainable_params)
        for sample in feature_matrix
    ])
    predicted_labels = (probabilities >= threshold).astype(float)

    return predicted_labels, probabilities


## 5. Hàm mất mát và gradient parameter-shift

Output của mạch là kỳ vọng `Z` trong khoảng `[-1, 1]`. Ta đổi sang xác suất lớp `1` bằng:

$$p(y=1|x) = \frac{1 - \langle Z \rangle}{2}$$

Sau đó dùng binary cross-entropy:

$$L = -\frac{1}{N}\sum_i y_i\log(p_i) + (1-y_i)\log(1-p_i)$$

Với các cổng quay lượng tử, đạo hàm theo một tham số có thể tính bằng **parameter-shift rule**:

$$\frac{\partial f(\theta)}{\partial \theta} = \frac{f(\theta + \pi/2) - f(\theta - \pi/2)}{2}$$

Trong notebook này, `f` là toàn bộ pipeline tính loss. Vì vậy, với mỗi tham số, ta chạy loss hai lần: một lần dịch lên `+pi/2`, một lần dịch xuống `-pi/2`.


In [ ]:
def binary_cross_entropy(true_labels, predicted_probabilities, epsilon=1e-8):
    """Compute binary cross-entropy with clipping for numerical stability."""
    clipped_probabilities = np.clip(predicted_probabilities, epsilon, 1 - epsilon)

    loss_per_sample = -(
        true_labels * np.log(clipped_probabilities)
        + (1 - true_labels) * np.log(1 - clipped_probabilities)
    )

    return np.mean(loss_per_sample)


def compute_loss(feature_matrix, true_labels, trainable_params):
    """Compute dataset loss for the current QNN parameters."""
    _, predicted_probabilities = predict_labels_and_probabilities(feature_matrix, trainable_params)
    return binary_cross_entropy(true_labels, predicted_probabilities)


def compute_accuracy(feature_matrix, true_labels, trainable_params):
    """Compute classification accuracy for the current QNN parameters."""
    predicted_labels, _ = predict_labels_and_probabilities(feature_matrix, trainable_params)
    return np.mean(predicted_labels == true_labels)


def parameter_shift_gradient(feature_matrix, true_labels, trainable_params):
    """Estimate d(loss)/d(theta_i) for every trainable parameter."""
    gradient = np.zeros_like(trainable_params)
    shift = np.pi / 2

    for param_index in range(len(trainable_params)):
        params_shifted_up = trainable_params.copy()
        params_shifted_down = trainable_params.copy()

        params_shifted_up[param_index] += shift
        params_shifted_down[param_index] -= shift

        loss_shifted_up = compute_loss(feature_matrix, true_labels, params_shifted_up)
        loss_shifted_down = compute_loss(feature_matrix, true_labels, params_shifted_down)

        gradient[param_index] = 0.5 * (loss_shifted_up - loss_shifted_down)

    return gradient


## 6. Huấn luyện QNN

Ta dùng gradient descent đơn giản:

$$\theta \leftarrow \theta - \eta \nabla_{\theta}L$$

Trong đó:

- `theta` là vector tham số của ansatz.
- `eta` là learning rate.
- `L` là hàm mất mát binary cross-entropy.

Vì mô phỏng state vector và tính parameter-shift khá tốn chi phí, dataset và số epoch được giữ nhỏ để chạy nhanh.


In [ ]:
def initialize_qnn_parameters(random_generator, n_params, scale=0.1):
    """Initialize trainable circuit angles near zero."""
    return random_generator.normal(loc=0.0, scale=scale, size=n_params)


def train_qnn(feature_matrix, true_labels, initial_params, learning_rate, epochs):
    """Train the QNN with full-batch gradient descent."""
    trainable_params = initial_params.copy()
    history = {'loss': [], 'accuracy': []}

    for epoch in range(1, epochs + 1):
        gradient = parameter_shift_gradient(feature_matrix, true_labels, trainable_params)
        trainable_params -= learning_rate * gradient

        epoch_loss = compute_loss(feature_matrix, true_labels, trainable_params)
        epoch_accuracy = compute_accuracy(feature_matrix, true_labels, trainable_params)

        history['loss'].append(epoch_loss)
        history['accuracy'].append(epoch_accuracy)

        if epoch == 1 or epoch % 5 == 0:
            print(f'Epoch {epoch:02d} | loss = {epoch_loss:.4f} | accuracy = {epoch_accuracy:.3f}')

    return trainable_params, history


initial_params = initialize_qnn_parameters(rng, N_TRAINABLE_PARAMS)
trained_params, training_history = train_qnn(
    features,
    labels,
    initial_params,
    learning_rate=0.35,
    epochs=45,
)


## 7. Đánh giá kết quả

Ta vẽ đường loss, accuracy qua từng epoch và biên quyết định của QNN trên mặt phẳng dữ liệu. Biên quyết định cho biết vùng nào mô hình dự đoán là lớp `0` hoặc lớp `1`.


In [ ]:
def plot_training_history(history):
    """Plot loss and accuracy curves from the training loop."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

    axes[0].plot(history['loss'], linewidth=2)
    axes[0].set_title('Training loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Binary cross-entropy')
    axes[0].grid(alpha=0.25)

    axes[1].plot(history['accuracy'], linewidth=2)
    axes[1].set_title('Training accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1.05)
    axes[1].grid(alpha=0.25)

    plt.tight_layout()
    plt.show()


plot_training_history(training_history)


In [ ]:
def make_decision_grid(min_value=-1, max_value=1, grid_size=80):
    """Create a 2D grid used to visualize the decision boundary."""
    axis_1 = np.linspace(min_value, max_value, grid_size)
    axis_2 = np.linspace(min_value, max_value, grid_size)
    grid_x, grid_y = np.meshgrid(axis_1, axis_2)
    grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]

    return grid_x, grid_y, grid_points


def plot_decision_boundary(feature_matrix, true_labels, trainable_params, feature_names):
    """Plot P(y = 1) over the feature plane and the 0.5 boundary."""
    grid_x, grid_y, grid_points = make_decision_grid()
    _, grid_probabilities = predict_labels_and_probabilities(grid_points, trainable_params)
    probability_surface = grid_probabilities.reshape(grid_x.shape)

    plt.figure(figsize=(5.4, 4.4))
    plt.contourf(grid_x, grid_y, probability_surface, levels=30, cmap='coolwarm', alpha=0.75)
    plt.colorbar(label='P(y = 1)')
    plt.scatter(feature_matrix[:, 0], feature_matrix[:, 1], c=true_labels, cmap='coolwarm', edgecolor='k', s=28)
    plt.contour(grid_x, grid_y, probability_surface, levels=[0.5], colors='black', linewidths=1.5)
    plt.title('QNN decision boundary on Iris')
    plt.xlabel(f'{feature_names[0]} scaled')
    plt.ylabel(f'{feature_names[1]} scaled')
    plt.grid(alpha=0.2)
    plt.show()


plot_decision_boundary(features, labels, trained_params, feature_names)

print(f'Final loss: {compute_loss(features, labels, trained_params):.4f}')
print(f'Final accuracy: {compute_accuracy(features, labels, trained_params):.3f}')


## 8. Tóm tắt mô hình

Mô hình QNN trong notebook này có cấu trúc:

```text
Input x = [petal length, petal width]
        |
        v
Quantum feature map
        |
        v
Parameterized quantum circuit / variational ansatz
        |
        v
Measure <Z>
        |
        v
Convert to probability P(y = 1)
        |
        v
Binary classification: setosa hoặc versicolor
```

Điểm cần nhớ:

- **Qubit** đóng vai trò không gian biểu diễn dữ liệu.
- **Feature map** giống lớp embedding, đưa dữ liệu cổ điển vào Hilbert space.
- **Ansatz có tham số** giống các layer học được trong neural network.
- **Measurement** biến thông tin lượng tử thành output cổ điển.
- **Parameter-shift** là cách tính gradient tự nhiên cho các cổng quay lượng tử.
- **Seed / random generator** giúp khởi tạo tham số giống nhau giữa các lần chạy để kết quả có thể tái lập.

Bạn có thể thử thay đổi `N_LAYERS`, `learning_rate`, `epochs`, hoặc thiết kế `build_feature_map` để quan sát mô hình học khác nhau như thế nào.
